# Compute pilot

Checks that the model trains on this GPU and measures what each part costs, so we can size the experiments before committing to them. This is the roadmap's Nov 12–15 milestone: **one browser episode, its reward, an environment reset, a real RL update, and an adapter save/reload.**

The notebook imports the shared code in `src/browser_agent/` and reads the same configs as real runs. That way the numbers carry over to SFT, RLOO and GRPO.

| Section | Measures |
| --- | --- |
| 1. Setup | GPU model, memory, bf16 support |
| 2. Load the model | base model with a LoRA adapter: load time, resident memory |
| 3. SFT steps | prompt lengths, seconds and peak memory per training step |
| 4. RL rollouts | one group of browser episodes: seconds per step, tokens |
| 5. RL update | one RLOO-style policy-gradient update on that group |
| 6. Save and reload | adapter round trip, then an episode through the normal `adapter_path` route |
| 7. Budget | browser interactions per GPU hour, and hours for a given budget |

**On Colab:** *Runtime → Change runtime type → GPU*, then *Runtime → Run all*. A T4 run takes about 10–15 minutes, most of it the model download.

**On a GCP VM:** `scripts/gcp/vm.sh pilot` runs this notebook on the VM without opening it. See [`docs/gcp.md`](../docs/gcp.md).

## 1. Setup

In [ ]:
# Which code to test. Use a branch name to try changes that aren't merged yet.
REPO_URL = "https://github.com/jasmineztruong8/browser-control.git"
BRANCH = "main"

# Copy the results (and the pilot adapter) to Google Drive. Colab's disk is wiped when the
# runtime disconnects.
SAVE_TO_DRIVE = False

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/browser-control")
    if not REPO_DIR.exists():
        !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -e ".[train]"
    !playwright install --with-deps chromium > /dev/null
else:
    # A GCP VM or a local checkout: the environment is already set up
    # (scripts/gcp/remote_setup.sh on a VM). Work from the repo root.
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO_DIR)

# A fresh editable install isn't always importable until the kernel restarts.
sys.path.insert(0, str(REPO_DIR / "src"))
!git log -1 --format="Testing commit %h: %s"

In [ ]:
import platform

import torch

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    GPU = {
        "name": props.name,
        "total_memory_gb": round(props.total_memory / 2**30, 1),
        "bf16_supported": torch.cuda.is_bf16_supported(),
    }
else:
    # Only for a quick local smoke test with a tiny model. Real numbers need a GPU.
    print("No GPU found. In Colab: Runtime > Change runtime type > GPU.")
    GPU = {
        "name": f"CPU ({platform.processor() or platform.machine()})",
        "total_memory_gb": None,
        "bf16_supported": False,
    }
GPU.update(torch=torch.__version__, cuda=torch.version.cuda)
GPU

## 2. Load the model

`CONFIG` is the config the pilot uses. `rloo.yaml` fits best because it has both the sampling settings and the RL settings (`training.group_size`, LoRA). Anything you'd change for a real run goes in `OVERRIDES`, using the same `section.key=value` syntax as `--set` on the command line. For example:

- `policy.load_in_4bit=false` measures full-precision weights
- `policy.model_name=Qwen/Qwen2.5-1.5B-Instruct` tries a smaller model
- `env.max_steps=8` or `observation.max_elements=40` shortens episodes or prompts

The pilot-only settings are the two counts below.

In [ ]:
CONFIG = "experiments/configs/rloo.yaml"
OVERRIDES = [
    # T4s have no bf16; fall back to fp16 there.
    f"policy.compute_dtype={'bfloat16' if GPU['bf16_supported'] else 'float16'}",
]
if not HAS_GPU:
    # Local smoke test: tiny random model, no 4-bit (bitsandbytes needs CUDA), float32 on CPU.
    OVERRIDES += [
        "policy.model_name=trl-internal-testing/tiny-Qwen2ForCausalLM-2.5",
        "policy.load_in_4bit=false",
        "policy.compute_dtype=float32",
        "policy.device_map=cpu",
        "policy.max_new_tokens=16",
    ]
# Extra overrides without editing the notebook, e.g. from scripts/gcp/vm.sh pilot KEY=VALUE.
OVERRIDES += os.environ.get("PILOT_OVERRIDES", "").split()

SFT_STEPS = 6  # training steps to time; the first is warm-up and is reported separately
RL_TASKS = 1  # tasks in the RL update; each runs training.group_size episodes

In [ ]:
import gc
import json
import statistics
import time
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager

from browser_agent.config import load_config, save_config
from browser_agent.rollout import make_env, make_policy, run_episode, summarize
from browser_agent.tasks import load_tasks

cfg = load_config(CONFIG, OVERRIDES)
RESULTS = {"gpu": GPU, "config": CONFIG, "overrides": OVERRIDES}
GB = 2**30


def sync():
    if HAS_GPU:
        torch.cuda.synchronize()


@contextmanager
def measure(name):
    """Time a block and record its peak GPU memory under RESULTS[name]."""
    sync()
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    record = {}
    start = time.perf_counter()
    yield record
    sync()
    record["seconds"] = round(time.perf_counter() - start, 2)
    if HAS_GPU:
        record["peak_memory_gb"] = round(torch.cuda.max_memory_allocated() / GB, 2)
        record["allocated_after_gb"] = round(torch.cuda.memory_allocated() / GB, 2)
    RESULTS[name] = record
    print(f"{name}: {record}")


# Playwright's sync API won't run on a thread with a running asyncio loop, and the notebook's
# main thread has one. All browser work therefore goes through this one worker thread
# (Playwright objects must also stay on the thread that created them).
browser_thread = ThreadPoolExecutor(max_workers=1)


def in_browser(fn, *args, **kwargs):
    return browser_thread.submit(fn, *args, **kwargs).result()


env = in_browser(make_env, cfg)
print(
    f"model={cfg.policy.model_name}  4bit={cfg.policy.load_in_4bit}  "
    f"dtype={cfg.policy.compute_dtype}  group_size={cfg.training.group_size}"
)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

with measure("model_load"):
    policy = make_policy(cfg)  # the same HFPolicy that rollouts and evaluation use

t = cfg.training
model = policy.model
if cfg.policy.load_in_4bit:
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=t.gradient_checkpointing
    )
elif t.gradient_checkpointing:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
model = get_peft_model(
    model,
    LoraConfig(
        r=t.lora.r,
        lora_alpha=t.lora.alpha,
        lora_dropout=t.lora.dropout,
        target_modules=t.lora.target_modules,
        task_type="CAUSAL_LM",
    ),
)
policy.model = model  # rollouts now generate with the (trainable) adapter
tok = policy.tokenizer
# SFT and RL use different learning rates (sft.yaml vs the RL config); section 5 switches.
SFT_LR = load_config("experiments/configs/sft.yaml").training.learning_rate
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=SFT_LR)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
RESULTS["model_load"].update(trainable_params=trainable, total_params=total)
print(f"trainable params: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")


def generation_mode():
    model.eval()
    model.config.use_cache = True  # gradient checkpointing turns the KV cache off


def training_mode():
    model.train()
    model.config.use_cache = False

## 3. SFT steps

The scripted policy plays each train task's demo in the browser. Every step becomes one example: the prompt the model would see, and the demo's action as the target. These are the same `(messages, model_output)` pairs the SFT trainer will use. The loss only counts the action tokens.

In [ ]:
demo_cfg = load_config(CONFIG, OVERRIDES + ["policy.name=scripted"])
scripted = make_policy(demo_cfg)
demo_tasks = load_tasks(
    demo_cfg.tasks.path, demo_cfg.tasks.split, demo_cfg.tasks.families, demo_cfg.tasks.limit
)

with measure("demo_collection") as rec:
    demos = in_browser(lambda: [run_episode(env, scripted, task, demo_cfg) for task in demo_tasks])
    rec.update(episodes=len(demos), success_rate=summarize(demos)["success_rate"])

examples = [
    (step.messages, step.model_output)
    for traj in demos
    if traj.success
    for step in traj.steps
    if step.action is not None
]
assert examples, "no successful demo steps to train on"


def chat_ids(messages):
    ids = tok.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True
    )["input_ids"]
    return list(ids)


def encode(messages, target):
    """Prompt + target ids, with the prompt masked out of the labels."""
    prompt = chat_ids(messages)
    answer = tok(target + tok.eos_token, add_special_tokens=False)["input_ids"]
    return prompt + answer, [-100] * len(prompt) + answer


def collate(pairs):
    encoded = [encode(m, a) for m, a in pairs]
    width = max(len(ids) for ids, _ in encoded)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for ids, labels in encoded:
        gap = width - len(ids)
        batch["input_ids"].append(ids + [pad] * gap)
        batch["attention_mask"].append([1] * len(ids) + [0] * gap)
        batch["labels"].append(labels + [-100] * gap)
    return {k: torch.tensor(v, device=model.device) for k, v in batch.items()}


prompt_lengths = [len(chat_ids(m)) for m, _ in examples]
RESULTS["prompt_tokens"] = {
    "examples": len(examples),
    "mean": round(statistics.mean(prompt_lengths)),
    "max": max(prompt_lengths),
}
print(
    f"{len(examples)} examples from {len(demos)} demos; prompt tokens: {RESULTS['prompt_tokens']}"
)

In [ ]:
batch_size = t.batch_size
step_seconds = []
training_mode()
with measure("sft_steps") as rec:
    for i in range(SFT_STEPS):
        batch = collate([examples[(i * batch_size + j) % len(examples)] for j in range(batch_size)])
        start = time.perf_counter()
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        sync()
        step_seconds.append(time.perf_counter() - start)
        print(f"step {i}: loss={loss.item():.3f}  {step_seconds[-1]:.2f}s")
    steady = step_seconds[1:] or step_seconds
    rec.update(
        batch_size=batch_size,
        first_step_s=round(step_seconds[0], 2),
        median_step_s=round(statistics.median(steady), 2),
        seconds_per_example=round(statistics.median(steady) / batch_size, 3),
    )

## 4. RL rollouts

One group of `training.group_size` sampled episodes per task, with the model choosing every action. This is the expensive part of RL, because every step is a generation plus a browser round trip.

The model isn't fine-tuned yet, so expect failures and invalid actions. An episode ends after `agent.max_consecutive_invalid` invalid actions in a row, so episodes here run shorter than they will after SFT. That's why the budget section works per browser step, not per episode.

In [ ]:
rl_tasks = load_tasks(cfg.tasks.path, cfg.tasks.split, cfg.tasks.families, RL_TASKS)
generation_mode()


def rollout_groups():
    return [[run_episode(env, policy, task, cfg) for _ in range(t.group_size)] for task in rl_tasks]


with measure("rl_rollouts") as rec:
    groups = in_browser(rollout_groups)
rollouts = [traj for group in groups for traj in group]
steps = [s for traj in rollouts for s in traj.steps]
generation_s = sum(s.latency_s for s in steps)
rec.update(
    episodes=len(rollouts),
    env_steps=len(steps),
    seconds_per_step=round(rec["seconds"] / len(steps), 2),
    generation_s_per_step=round(generation_s / len(steps), 2),
    browser_s_per_step=round((rec["seconds"] - generation_s) / len(steps), 2),
    mean_prompt_tokens=round(statistics.mean(s.prompt_tokens or 0 for s in steps)),
    mean_completion_tokens=round(statistics.mean(s.completion_tokens or 0 for s in steps)),
    **{k: summarize(rollouts)[k] for k in ("success_rate", "mean_reward", "invalid_action_rate")},
)
print(rec)
for traj in rollouts:
    print(f"{traj.task_id}: reward={traj.reward:.2f} steps={len(traj.steps)} end={traj.terminated}")
print("first model output:", repr(rollouts[0].steps[0].model_output))

## 5. RL update

An RLOO update on the groups from section 4. Each episode's advantage is its reward minus the mean reward of the other episodes in its group, and the loss is `-advantage × log p(actions)`. Steps are back-propagated one at a time, so peak memory reflects the longest single step, as it will in the real trainer.

If every episode in a group got the same reward (likely before SFT), the advantages are 0 and the gradient is zero. The time and memory are still what a real update costs.

In [ ]:
def action_logprob(messages, output):
    """Sum of log-probs of the model's output tokens, given the prompt."""
    ids, labels = encode(messages, output)
    input_ids = torch.tensor([ids], device=model.device)
    labels = torch.tensor([labels], device=model.device)
    logits = model(input_ids=input_ids).logits[:, :-1].float()
    targets = labels[:, 1:]
    mask = targets != -100
    logp = torch.log_softmax(logits, dim=-1).gather(-1, targets.clamp(min=0).unsqueeze(-1))
    return (logp.squeeze(-1) * mask).sum()


for group in optimizer.param_groups:
    group["lr"] = t.learning_rate  # the RL config's learning rate
training_mode()
with measure("rl_update") as rec:
    sequences, longest = 0, 0
    advantages = []
    for group in groups:
        rewards = torch.tensor([traj.reward for traj in group])
        baseline = (rewards.sum() - rewards) / (len(group) - 1)
        for traj, adv in zip(group, (rewards - baseline).tolist(), strict=True):
            advantages.append(adv)
            for step in traj.steps:
                loss = -adv * action_logprob(step.messages, step.model_output) / len(rollouts)
                loss.backward()
                sequences += 1
                longest = max(longest, (step.prompt_tokens or 0) + (step.completion_tokens or 0))
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    rec.update(sequences=sequences, longest_sequence_tokens=longest)
rec["seconds_per_sequence"] = round(rec["seconds"] / max(sequences, 1), 2)
print(rec)
print("advantages:", [round(a, 2) for a in advantages])

## 6. Save and reload the adapter

This saves the adapter and frees the model. It then loads the adapter the way evaluation will, through `policy.adapter_path`, and checks that it gives the same next-token logits as before saving. Last, it runs one episode with the reloaded policy.

In [ ]:
from peft import PeftModel, get_peft_model_state_dict

ADAPTER_DIR = Path(t.output_dir) / "pilot"
probe = collate([examples[0]])
generation_mode()
with torch.no_grad():
    logits_before = model(input_ids=probe["input_ids"]).logits[0, -1].float().cpu()
saved_weights = {k: v.detach().float().cpu() for k, v in get_peft_model_state_dict(model).items()}

with measure("adapter_save") as rec:
    model.save_pretrained(ADAPTER_DIR)
    rec["size_mb"] = round(
        sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 2**20, 1
    )

# Drop every reference to the trained model so its memory is freed before reloading.
model = policy = optimizer = batch = loss = None
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()

reload_cfg = load_config(CONFIG, OVERRIDES + [f"policy.adapter_path={ADAPTER_DIR}"])
with measure("adapter_reload"):
    policy = make_policy(reload_cfg)
with torch.no_grad():
    logits_after = policy.model(input_ids=probe["input_ids"]).logits[0, -1].float().cpu()

assert isinstance(policy.model, PeftModel), "policy.adapter_path didn't load a LoRA adapter"
loaded_weights = get_peft_model_state_dict(policy.model)
assert loaded_weights.keys() == saved_weights.keys(), "reloaded adapter has different layers"
weight_diff = max(
    (saved_weights[k] - v.detach().float().cpu()).abs().max().item()
    for k, v in loaded_weights.items()
)
# Logits are reported but not checked strictly: training upcasts layer norms and the output
# layer to fp32 (prepare_model_for_kbit_training) while inference runs them in bf16, which
# alone moves logits of a 3B model by about 1.
diff = (logits_before - logits_after).abs().max().item()
same_top = logits_before.argmax().item() == logits_after.argmax().item()
RESULTS["adapter_reload"].update(
    max_weight_diff=weight_diff, max_logit_diff=round(diff, 4), same_top_token=same_top
)
print(f"max LoRA weight difference {weight_diff:.2e}")
print(f"max logit difference {diff:.4f}, same top token: {same_top}")
assert weight_diff < 1e-3, "the reloaded LoRA weights don't match the ones that were saved"

traj = in_browser(run_episode, env, policy, rl_tasks[0], reload_cfg)
RESULTS["reloaded_episode"] = {
    "task": traj.task_id,
    "steps": len(traj.steps),
    "reward": traj.reward,
    "terminated": traj.terminated,
}
print("episode with the reloaded adapter:", RESULTS["reloaded_episode"])

## 7. Budget

One RL browser interaction costs one rollout step (generation plus browser), plus its share of the update. From that come interactions per GPU hour, and the GPU hours for a given interaction budget per RL condition. `training.max_env_interactions` is the per-condition budget that the roadmap requires to be matched across RLOO, GRPO and shaped GRPO.

SFT cost is the time per example times the number of examples per epoch. PR #4's 65 train tasks give roughly 4–6 steps each.

These numbers come from an untrained model on a handful of tasks, so treat them as order-of-magnitude. The roadmap also reserves about 20% of credits for failures and reruns.

In [ ]:
from IPython.display import Markdown, display

r = RESULTS
per_interaction_s = r["rl_rollouts"]["seconds_per_step"] + r["rl_update"]["seconds_per_sequence"]
interactions_per_hour = 3600 / per_interaction_s
RL_CONDITIONS = 3  # B (RLOO), C (GRPO), D (GRPO + subgoal shaping)

budgets = [t.max_env_interactions] if t.max_env_interactions else [1_000, 2_500, 5_000, 10_000]
r["budget"] = {
    "seconds_per_interaction": round(per_interaction_s, 2),
    "interactions_per_gpu_hour": round(interactions_per_hour),
    "rl_gpu_hours": {
        n: {
            "per_condition": round(n / interactions_per_hour, 1),
            "all_conditions": round(RL_CONDITIONS * n / interactions_per_hour, 1),
        }
        for n in budgets
    },
    "sft_seconds_per_example": r["sft_steps"]["seconds_per_example"],
}

peak = max(v.get("peak_memory_gb", 0) for v in r.values() if isinstance(v, dict))
lines = [
    f"**GPU:** {GPU['name']} ({GPU['total_memory_gb']} GB)  ",
    f"**Model:** {cfg.policy.model_name}, 4-bit={cfg.policy.load_in_4bit}, "
    f"{cfg.policy.compute_dtype}, LoRA r={t.lora.r}",
    "",
    "| Measurement | Value |",
    "| --- | --- |",
    f"| Peak GPU memory (any stage) | {peak} GB |",
    f"| Model load | {r['model_load']['seconds']} s |",
    f"| Prompt length (mean / max) | {r['prompt_tokens']['mean']} / {r['prompt_tokens']['max']} tokens |",
    f"| SFT step (batch {r['sft_steps']['batch_size']}) | {r['sft_steps']['median_step_s']} s, "
    f"peak {r['sft_steps'].get('peak_memory_gb', '-')} GB |",
    f"| Rollout step (generation + browser) | {r['rl_rollouts']['generation_s_per_step']} + "
    f"{r['rl_rollouts']['browser_s_per_step']} s |",
    f"| RL update per sequence | {r['rl_update']['seconds_per_sequence']} s, "
    f"peak {r['rl_update'].get('peak_memory_gb', '-')} GB |",
    f"| Adapter save / reload | {r['adapter_save']['size_mb']} MB, weights match: "
    f"{r['adapter_reload']['max_weight_diff'] < 1e-3}, same top token: "
    f"{r['adapter_reload']['same_top_token']} |",
    f"| **RL interactions per GPU hour** | **{round(interactions_per_hour)}** |",
]
for n, hours in r["budget"]["rl_gpu_hours"].items():
    lines.append(
        f"| {n:,} interactions | {hours['per_condition']} h per condition, "
        f"{hours['all_conditions']} h for B+C+D |"
    )
SUMMARY = "\n".join(lines)
display(Markdown(SUMMARY))

In [ ]:
from datetime import datetime

run_dir = Path(cfg.run.output_dir) / f"{datetime.now():%Y%m%d-%H%M%S}-pilot"
save_config(cfg, run_dir / "config.yaml")
(run_dir / "pilot.json").write_text(json.dumps(RESULTS, indent=2, default=str))
(run_dir / "summary.md").write_text(SUMMARY + "\n")
print(f"Saved to {run_dir}")

if SAVE_TO_DRIVE and IN_COLAB:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/browser-control") / run_dir.name
    shutil.copytree(run_dir, dest)
    shutil.copytree(ADAPTER_DIR, dest / "adapter")
    print(f"Copied to Drive: {dest}")

policy.close()
in_browser(env.close)
browser_thread.shutdown()

## Next

Add the summary table above to `results/` (or the team's notes), along with the GPU it came from. If peak memory came close to the GPU's total, or the hours look too high, try these in `OVERRIDES` and rerun:

1. `policy.load_in_4bit=true`, if it was turned off
2. shorter prompts: `observation.max_elements`, `observation.max_page_text_chars`, `agent.history_length`
3. `training.group_size=2` or a smaller `training.batch_size`
4. a smaller model, used for every condition